In [1]:
import pandas as pd

In [5]:
years = [2019, 2020, 2021, 2022]
sheets = ["Table 111", "Table 112", "Table 113", "Table 114"]
dfs = []
for index, sheet in enumerate(sheets):
    dfs.append(pd.read_excel("../AGRICULTURE/anambra_2022_year_book.xlsx", sheet_name=sheets[index]))

lga_df = pd.read_csv("../Geography/Anambra_LGAs.csv")
lga_df.head()

,lga_id,lga_name,zone,AREA_SQ_KM,HIGHLANDS_percent,PLAIN_LANDS_percent
0,1,ONITSHA SOUTH,ONITSHA,196,20,48
1,2,ANAMBRA WEST,OTUOCHA,613,15,21
2,3,IDEMILI SOUTH,OGIDI,145,20,25
3,4,NNEWI SOUTH,NNEWI,176,34,38
4,5,OYI,OGIDI,331,10,40


In [6]:
for index, df in enumerate(dfs):
    dfs[index] = dfs[index].fillna(0)
    dfs[index].replace(['NA', 'NIL', '-'], 0, inplace=True)

dfs[1].head()

,LGA,EROSION SITE,TYPE OF EROSION,TYPE OF INTERVENTION,AREA (HECTARES),AVERAGE DEPTH (METRES)
0,AGUATA,0,0,0,0.0,0.0
1,ANAMBRA EAST,0,0,0,0.0,0.0
2,ANAMBRA WEST,0,0,0,0.0,0.0
3,ANAOCHA,0,0,0,0.0,0.0
4,AWKA NORTH,0,0,0,0.0,0.0


In [ ]:
columns = ['id', 'lga_id', 'erosion_site']
site_df = pd.DataFrame(columns=columns)
for i , df in enumerate(dfs):
    for index, row in df.iterrows():
        if row["EROSION SITE"] in (0, '0'): continue

        if(row["LGA"] not in (0, '0')): lga=row["LGA"]

        if(len(lga_df.loc[lga_df['lga_name'] == lga, "lga_id"]) == 0):
            print(row["LGA"])
        # print(row)
        transformed_data = [{
            'lga_id': lga_df.loc[lga_df['lga_name'] == lga, "lga_id"].values[0],
            'erosion_site': row['EROSION SITE'].replace("\n", " ").strip()
        },]   

        site_df = pd.concat([site_df, pd.DataFrame(transformed_data)], ignore_index=True)

site_df = site_df.drop_duplicates()
site_df['id'] = site_df.index
site_df.head(50)

,id,lga_id,erosion_site
0,0,20,"Amachalla, Awka"
1,1,20,"Federal High Court/Ekwueme Square, Awka (Palli..."
2,2,20,Neroz Plaza/ St Thomas Aquinas
3,3,8,Ugamuma-Obosi Obosi
4,4,8,Ire-Obosi
5,5,8,Nkpor-Flyover
6,6,8,Abidi-Umuoji
7,7,8,Ikenga-Ogidi
8,8,3,Ojoto
9,9,11,Abagana


In [47]:
site_df.to_excel("erosion_sites.xlsx", index=False)
print("Done!")

Done!


In [64]:
columns = ['id', 'intervention']
intervention_df = pd.DataFrame(columns=columns)
for i , df in enumerate(dfs):
    for index, row in df.iterrows():
        if row["TYPE OF INTERVENTION"] in (0, '0'): continue

        if(row["LGA"] not in (0, '0')): lga=row["LGA"]

        if(len(lga_df.loc[lga_df['lga_name'] == lga, "lga_id"]) == 0):
            print(row["LGA"])
        # print(row)
        interventions = row["TYPE OF INTERVENTION"].replace('&', '-').replace(',', '-').split('-')
        for intervention in interventions:
            transformed_data = [{
                'intervention': intervention.replace("\n", " ").strip()
            },]   

            intervention_df = pd.concat([intervention_df, pd.DataFrame(transformed_data)], ignore_index=True)

intervention_df = intervention_df.drop_duplicates()
intervention_df['id'] = intervention_df.index
intervention_df.head(50)
print(intervention_df['intervention'])

0    Livelihood Enhancement
1         Civil Engineering
2            Bioremediation
Name: intervention, dtype: object


In [49]:
intervention_df.to_excel("intervention_type.xlsx", index=False)
print(0)

0


In [50]:
columns = ['id', 'lga_id', 'year', 'erosion_site_id', 'type_of_erosion', 'area_in_hectares', 'avg_depth_in_metres']

new_df = pd.DataFrame(columns=columns)

for i, df in enumerate(dfs):
    year = years[i]
    for index, row in df.iterrows():
        if row["EROSION SITE"] in (0, '0'): continue
        if(row["LGA"] not in (0, '0')): lga=row["LGA"]
        
        if(len(lga_df.loc[lga_df['lga_name'] == lga, "lga_id"]) == 0):
            print(row["LGA"])
        if(len(site_df.loc[site_df['erosion_site'] == row["EROSION SITE"].strip().replace("\n", " "), "id"].values)==0): print(row["EROSION SITE"])
        # print(row)
        transformed_data = [{
            'lga_id': lga_df.loc[lga_df['lga_name'] == lga, "lga_id"].values[0],
            'year': year,
            'erosion_site_id': site_df.loc[site_df['erosion_site'] == row["EROSION SITE"].strip().replace("\n", " "), "id"].values[0],
            'type_of_erosion': row["TYPE OF EROSION"],
            'area_in_hectares': row["AREA (HECTARES)"],
            'avg_depth_in_metres': row["AVERAGE DEPTH (METRES)"]
        },]   

        new_df = pd.concat([new_df, pd.DataFrame(transformed_data)], ignore_index=True)

new_df['id']=new_df.index
new_df.head(22)

C:\Users\USER\AppData\Local\Temp\ipykernel_7696\46062118.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_df = pd.concat([new_df, pd.DataFrame(transformed_data)], ignore_index=True)


,id,lga_id,year,erosion_site_id,type_of_erosion,area_in_hectares,avg_depth_in_metres
0,0,20,2019,0,Gully,2.00,20.0
1,1,20,2019,1,Gully,3.49,10.8
2,2,20,2019,2,Gully,1.30,15.0
3,3,8,2019,3,Gully,10.88,8.5
4,4,8,2019,4,Gully,16.60,23.0
5,5,8,2019,5,Gully,5.40,12.0
6,6,8,2019,6,Gully,2.65,7.7
7,7,8,2019,7,Gully,3.00,6.0
8,8,3,2019,8,Gully,7.70,22.0
9,9,11,2019,9,Gully,4.80,12.0


In [51]:
new_df.to_excel("erosion_site_intervention.xlsx", index=False)
print("Done!")

Done!


In [62]:
columns = ['id', 'intervention_type_id', 'erosion_site_intervention_id']
erosion_site_intervention_by_type_df = pd.DataFrame(columns=columns)
for i , df in enumerate(dfs):
    for index, row in df.iterrows():
        if row["TYPE OF INTERVENTION"] in (0, '0'): continue
        print("local government area", row["LGA"])
        if(row["LGA"] not in (0, '0')): lga=row["LGA"]

        lga_id= lga_df.loc[lga_df['lga_name'] == lga, "lga_id"].values[0]

        interventions = row["TYPE OF INTERVENTION"].replace('&', '-').replace(',', '-').replace('\n', ' ').split('-')
        site_id = site_df.loc[site_df["erosion_site"] == row["EROSION SITE"].replace("\n", " ").strip(), 'id'].values[0]

        erosion_site_intervention_id = new_df.loc[(new_df["lga_id"] == lga_id) & (new_df["erosion_site_id"]==site_id) , "id"].values[0]

        for intervention in interventions:
            print(intervention)
            intervention_id = intervention_df.loc[intervention_df["intervention"]==intervention.replace("\n", " ").strip(), 'id'].values[0]

            transformed_data = [{
                'intervention_type_id': intervention_id,
                'erosion_site_intervention_id': erosion_site_intervention_id
            },]   

            erosion_site_intervention_by_type_df = pd.concat([erosion_site_intervention_by_type_df, pd.DataFrame(transformed_data)], ignore_index=True)

erosion_site_intervention_by_type_df = erosion_site_intervention_by_type_df.drop_duplicates()
erosion_site_intervention_by_type_df['id'] = erosion_site_intervention_by_type_df.index
erosion_site_intervention_by_type_df.head(50)

local government area AWKA SOUTH
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area 0
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area 0
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area IDEMILI NORTH
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area 0
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area 0
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area 0
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area 0
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area IDEMILI SOUTH
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area NJIKOKA
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area 0
Livelihood Enhancement
 Civil Engineering 
 Bioremediation
local government area NNEWI NORTH
Livelihood Enh

,id,intervention_type_id,erosion_site_intervention_id
0,0,0,0
1,1,1,0
2,2,2,0
3,3,0,1
4,4,1,1
5,5,2,1
6,6,0,2
7,7,1,2
8,8,2,2
9,9,0,3


In [63]:
erosion_site_intervention_by_type_df.to_excel("erosion_site_intervention_by_type.xlsx", index=False)
print(0)

0
